In [92]:
import numpy as np
from itertools import product
from gen.gen_utilities import permutation_symbol as perm

In [93]:
geom = {'L': 1,   # length of beam, m
        'b': 0.05, # height of beam, m,
        'h': 0.05, # heaight of beam m
        'A1': 0.05, # area of beam in m^2
        'A2': 0.05, # area of beam in m^2
        'A3': 0.0025, # cross-sectional area of beam in m^2
        'I1': 5.20833E-7, # moment of inertia m^4, b*h^3/12
        'I2': 5.20833E-7, # moment of inertia m^4,  inertia about E2xE2
        'J': 1.04166E-6, # polar moment of inertia m^4, pi*(D^4-d^4)/32, inertia about E3xE3, J=I1+I2
        }

material = {'E': 210E9, # Youngs modulus, Pa
                'rho': 7850, # density of steel, kg/m^3
                'G': 80E9, # shear modulus, Pa
                'Ks': 1.2E4 # shear correction coefficient, unitless
                }

S0 = np.array([0.5,0.5])
S1 = np.array([-2,2])
S00 = np.outer(S0,S0)
S11 = np.outer(S1,S1)
S10 = np.outer(S1,S0)
               
appForceGP = np.array([0,0,0], dtype=np.float64)
appMomentGP = np.array([0,0,0], dtype=np.float64)
matModFGP = np.diag([material['G'] * geom['A1'] * material['Ks'], 
                     material['G'] * geom['A2'] * material['Ks'], 
                     material['E'] * geom['A3']
                     ])

matModMGP = np.diag([material['E'] * geom['I1'], 
                     material['E'] * geom['I2'], 
                     material['G'] * geom['J'] 
                     ])

omegaGP = np.array([[[0,0,0]],[[0,0,0]]],dtype=np.float64)

In [94]:
x = 9.52380952e-06
phi = np.array([[0,0,(1-x)/2],[0,0,(1-x)]],dtype=np.float64)
dphio = np.dot(S1,phi)
rotation = np.array([[1,0,0],[0,1,0],[0,0,1]], dtype=np.float64)
gammaGP = dphio - (rotation @ np.array([0,0,1]))

forceGP = matModFGP @ gammaGP
momentGP = matModMGP @ omegaGP[0,0,:]

effecWt = 0.5


In [98]:
gammaGP

array([ 0.00000000e+00,  0.00000000e+00, -9.52380952e-06])

In [96]:
coeffSME = {f'K{i}{j}':np.zeros(shape=[2,2],dtype=np.float64) for i,j in product(range(6), repeat=2)} # memory allocation for constants of elemental stiffness matrices
K11a3b3 = 0.0

for a in range(3): # a=alpha, b=beta as notes in reference material
    for b in range(3):
        coeffSME[f'K{a}{b}'][:,:] = matModFGP[a,b] * S11 * effecWt
        # print(f'coeffSME[K{a}{b}]=',coeffSME[f'K{a}{b}'])
        K11a3b3 = matModMGP[a,b]
        K10ab3 = 0.0
        K01a3b = 0.0
        K00a3b3_0 = 0.0
        K00a3b3_1 = 0.0
        K10a3b3 = 0.0
        for k in range(3):
            for l in range(3):
                K10ab3 += perm()[k,b,l] * dphio[k]*matModFGP[a,l] - perm()[k,b,a]*forceGP[k]
                K01a3b += perm()[k,a,l] * dphio[k]*matModFGP[b,l] + perm()[k,b,a]*forceGP[k]
                K10a3b3 += - perm()[k,b,a]*momentGP[k]
                for m in range(3):
                    K00a3b3_1 += perm()[a,k,l]*perm()[m,b,l]*dphio[k]*forceGP[m]
                    for n in range(3):
                        K00a3b3_0 += perm()[k,a,l]*perm()[m,b,n]*dphio[k]*dphio[m]*matModFGP[l,n]
        # print('a=',a,'b=',b,'K01a3b=',K01a3b,'K10ab3=',K10ab3)
        coeffSME[f'K{a}{b+3}'][:,:] = K10ab3 * S10 * effecWt
        # print(f'coeffSME[K{a}{b+3}]=',coeffSME[f'K{a}{b+3}'])
        coeffSME[f'K{a+3}{b}'][:,:] = K01a3b * S10.T * effecWt
        # print(f'coeffSME[K{a+3}{b}]=',coeffSME[f'K{a+3}{b}'])
        coeffSME[f'K{a+3}{b+3}'][:,:] = (K11a3b3 * S11 + K10a3b3*S10 + (K00a3b3_0+K00a3b3_1)*S00) * effecWt
        # print(f'coeffSME[K{a+3}{b+3}]=',coeffSME[f'K{a+3}{b+3}'])

# print('coeffSME=',coeffSME)      
        

In [97]:
coeffCVE = {f'F{i}':np.zeros(shape=[2],dtype=np.float64) for i in range(6)} # memory allocation for constants of elemental stiffness matrices
F0a = 0.0
F1a = 0.0
F1a3 = 0.0
for a in range(3):
    F0a = appForceGP[a]
    F1a = -forceGP[a]
    coeffCVE[f'F{a}'][:] = (F0a*S0 + F1a*S1) * effecWt
    print(f'coeffCVE[F{a}]=',coeffCVE[f'F{a}'])
    F1a3 = -momentGP[a]
    F0a3 = 0.0
    for k in range(3):
        for l in range(3):
            F0a3 += perm()[a,l,k] * forceGP[k] * dphio[l] + appMomentGP[a]
    coeffCVE[f'F{a+3}'][:] = (F0a3*S0 + F1a3*S1) * effecWt
    print(f'coeffCVE[F{a+3}]=',coeffCVE[f'F{a+3}'])
    




coeffCVE[F0]= [0. 0.]
coeffCVE[F3]= [0. 0.]
coeffCVE[F1]= [0. 0.]
coeffCVE[F4]= [0. 0.]
coeffCVE[F2]= [-4999.99999801  4999.99999801]
coeffCVE[F5]= [0. 0.]


In [77]:
dphio

array([0.        , 0.        , 0.99999048])

In [78]:
gammaGP

array([ 0.00000000e+00,  0.00000000e+00, -9.52380952e-06])

In [79]:
perm()[:,1,1]

array([0., 0., 0.])